# KooChemicalSimulation - Getting Started

Phase 59: Jupyter Notebook Support

This tutorial introduces the basic features of the KooChemicalSimulation Python interface.

## Installation

```bash
pip install koolab
```

Or build from source:
```bash
cd KooChemicalSimulation
cmake -DENABLE_PYTHON=ON ..
make && sudo make install
```

## Import Library

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import koolab as koo

print(f"KooLab version: {koo.__version__}")
koo.print_info()

## 1. Mesh Generation

In [ ]:
# Create a simple rectangular mesh
mesh = koo.create_rectangular_mesh(0, 0, 1, 1, 10, 10)

print(f"Nodes: {mesh.num_nodes()}")
print(f"Elements: {mesh.num_elements()}")

# Convert to NumPy arrays
nodes, elements = koo.numpy_utils.mesh_to_arrays(mesh)
print(f"\nNode array shape: {nodes.shape}")
print(f"Number of elements: {len(elements)}")

In [ ]:
# Visualize mesh
plt.figure(figsize=(8, 8))
plt.scatter(nodes[:, 0], nodes[:, 1], c='blue', s=20, alpha=0.6)
plt.xlabel('X')
plt.ylabel('Y')
plt.title('Rectangular Mesh')
plt.axis('equal')
plt.grid(True, alpha=0.3)
plt.show()

## 2. Chemical Species

In [ ]:
# Create chemical species
h2 = koo.Species("H2")
h2.molar_mass = 0.002  # kg/mol
h2.set_composition({"H": 2})

o2 = koo.Species("O2")
o2.molar_mass = 0.032  # kg/mol
o2.set_composition({"O": 2})

h2o = koo.Species("H2O")
h2o.molar_mass = 0.018  # kg/mol
h2o.set_composition({"H": 2, "O": 1})

print(h2)
print(o2)
print(h2o)

## 3. Chemical Reactions

In [ ]:
# Create reaction: 2H2 + O2 => 2H2O
rxn = koo.Reaction()
rxn.add_reactant("H2", 2.0)
rxn.add_reactant("O2", 1.0)
rxn.add_product("H2O", 2.0)

print(f"Reaction: {rxn}")
print(f"Reactants: {rxn.get_reactants()}")
print(f"Products: {rxn.get_products()}")

In [ ]:
# Or parse from string
rxn2 = koo.parse_reaction("CH4 + 2O2 => CO2 + 2H2O")
print(f"Parsed reaction: {rxn2}")

## 4. Arrhenius Rate Constants

In [ ]:
# Arrhenius rate: k(T) = A * T^beta * exp(-Ea/RT)
rate = koo.ArrheniusRate(A=1e13, beta=0.0, Ea=150000)

print(f"Arrhenius parameters: {rate}")

# Evaluate at different temperatures
temperatures = np.linspace(500, 2000, 100)
rate_constants = np.array([rate(T) for T in temperatures])

# Plot
plt.figure(figsize=(10, 6))
plt.semilogy(temperatures, rate_constants, linewidth=2)
plt.xlabel('Temperature (K)', fontsize=12)
plt.ylabel('Rate constant k(T)', fontsize=12)
plt.title('Arrhenius Rate Constant', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.show()

## 5. GPU Information

In [ ]:
# Check GPU availability
gpu_info = koo.gpu.get_gpu_info()
print(f"GPU devices: {gpu_info['device_count']}")
print(f"GPU runtime: {gpu_info['runtime']}")
print(f"CUDA enabled: {gpu_info['cuda_enabled']}")
print(f"HIP enabled: {gpu_info['hip_enabled']}")

if gpu_info['device_count'] > 0:
    dev = koo.Device.get_device(0)
    props = dev.get_properties()
    print(f"\nGPU 0: {props.name}")
    print(f"Memory: {props.total_memory / 1e9:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")

## 6. Simple Reaction-Diffusion Example

In [ ]:
# Simple 1D diffusion simulation (conceptual)
nx = 100
L = 1.0
dx = L / (nx - 1)
x = np.linspace(0, L, nx)

# Initial condition: Gaussian pulse
C0 = np.exp(-100 * (x - 0.5)**2)

# Diffusion coefficient
D = 0.01
dt = 0.0001
nt = 1000

# Simple explicit diffusion
C = C0.copy()
snapshots = [C0.copy()]
snapshot_times = [0]

for n in range(nt):
    C_new = C.copy()
    for i in range(1, nx-1):
        C_new[i] = C[i] + D * dt / dx**2 * (C[i+1] - 2*C[i] + C[i-1])
    C = C_new
    
    if n % 200 == 0:
        snapshots.append(C.copy())
        snapshot_times.append((n+1) * dt)

# Plot evolution
plt.figure(figsize=(12, 6))
for i, (C_snap, t) in enumerate(zip(snapshots, snapshot_times)):
    plt.plot(x, C_snap, label=f't = {t:.4f}', linewidth=2)

plt.xlabel('Position', fontsize=12)
plt.ylabel('Concentration', fontsize=12)
plt.title('1D Diffusion Evolution', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

In this tutorial, we covered:
1. ✅ Mesh generation and visualization
2. ✅ Chemical species creation
3. ✅ Reaction definition and parsing
4. ✅ Arrhenius rate constants
5. ✅ GPU information
6. ✅ Simple diffusion example

Next tutorials:
- **02_advanced_chemistry.ipynb** - Complex reaction networks
- **03_gpu_acceleration.ipynb** - GPU-accelerated simulations
- **04_visualization.ipynb** - Advanced plotting